In [3]:
import pandas as pd
from tqdm import tqdm

agg_full = pd.read_csv("agg_full_max_5.csv", parse_dates=['order_date'])
print(agg_full.shape)
print(agg_full.columns)

(21132, 556)
Index(['order_date', 'coating_id', 'total_fill_rate', 'mean_fill_rate',
       'median_fill_rate', 'min_fill_rate', 'max_fill_rate', 'std_fill_rate',
       'total_chf', 'mean_chf',
       ...
       'rolling_mean_num_customers_nth_weekday_last_4_quarters',
       'rolling_mean_num_customers_nth_weekday_last_2_years',
       'rolling_mean_days_since_last_order_nth_weekday_last_3_months',
       'rolling_mean_days_since_last_order_nth_weekday_last_12_months',
       'rolling_mean_days_since_last_order_nth_weekday_last_4_quarters',
       'rolling_mean_days_since_last_order_nth_weekday_last_2_years',
       'rolling_mean_biggest_customer_orders_by_volume_nth_weekday_last_3_months',
       'rolling_mean_biggest_customer_orders_by_volume_nth_weekday_last_12_months',
       'rolling_mean_biggest_customer_orders_by_volume_nth_weekday_last_4_quarters',
       'rolling_mean_biggest_customer_orders_by_volume_nth_weekday_last_2_years'],
      dtype='object', length=556)


In [4]:
import numpy as np
from tsfresh.feature_extraction import extract_features, MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute
from sklearn.preprocessing import StandardScaler

fc_parameters = MinimalFCParameters()
window_lengths = [3]# [3,5,10,21,63]  # You can add more
base_features = [ # 'total_fill_rate']
    'total_fill_rate', 'total_chf', 'num_orders', 'num_customers',
    'num_industries', 'num_product_families'
]

def batch_rolling_windows(df, base_feature, window):
    """
    Collect all rolling window slices for all (coating_id, order_date) in one DataFrame,
    assigning a unique window_id to each for TSFRESH extraction.
    """
    all_windows = []
    for coating_id, group in tqdm(df.groupby('coating_id'), desc=f"Coating {base_feature}, win={window}"):
        group = group.sort_values('order_date').reset_index(drop=True)
        if len(group) >= window:
            for i in range(window-1, len(group)):
                window_slice = group.iloc[i-window+1:i+1][['coating_id', 'order_date', base_feature]].copy()
                # Assign unique window_id (coating_id + date of window end)
                window_slice['window_id'] = f"{coating_id}_{group.iloc[i]['order_date']}"
                window_slice['time'] = window_slice['order_date']
                window_slice['id'] = window_slice['window_id']  # For TSFRESH
                window_slice['value'] = window_slice[base_feature]
                all_windows.append(window_slice[['id', 'time', 'value']])
    if all_windows:
        return pd.concat(all_windows, ignore_index=True)
    else:
        return pd.DataFrame(columns=['id', 'time', 'value'])

all_tsfresh = []

for base_feature in base_features:
    for window in window_lengths:
        print(f"\nProcessing: {base_feature}, window={window}")
        windowed_df = batch_rolling_windows(agg_full, base_feature, window)
        if not windowed_df.empty:
            feats = extract_features(
                windowed_df,
                column_id='id',
                column_sort='time',
                column_value='value',
                default_fc_parameters=fc_parameters,
                n_jobs=4,
                disable_progressbar=False
            )
            # Extract coating_id and order_date from id
            feats = feats.reset_index()
            # After feats = feats.reset_index()
            id_col = 'id' if 'id' in feats.columns else feats.columns[0]
            feats[['coating_id', 'order_date']] = feats[id_col].astype(str).str.split('_', n=1, expand=True)
            feats['order_date'] = pd.to_datetime(feats['order_date'])
            feats = feats.drop(columns=[id_col])
            # Prefix feature names
            feats = feats.add_prefix(f"{base_feature}_w{window}_")
            feats = feats.rename(columns={f"{base_feature}_w{window}_coating_id": "coating_id",
                                         f"{base_feature}_w{window}_order_date": "order_date"})
            all_tsfresh.append(feats)

if all_tsfresh:
    tsfresh_rolling = all_tsfresh[0]
    for df in all_tsfresh[1:]:
        tsfresh_rolling = pd.merge(tsfresh_rolling, df, on=['coating_id', 'order_date'], how='outer')
    
    tsfresh_rolling.to_csv("tsfresh_results_3_not_imputed.csv", index=False)
    # Impute missing values
    numeric_cols = tsfresh_rolling.select_dtypes(include=[np.number]).columns
    # tsfresh_rolling[numeric_cols] = impute(tsfresh_rolling[numeric_cols])
    for col in numeric_cols:
        tsfresh_rolling[col] = tsfresh_rolling[col].fillna(0)
    
    tsfresh_rolling = tsfresh_rolling.sort_values(['coating_id', 'order_date']).reset_index(drop=True)
    print(f"Extracted {len(tsfresh_rolling.columns)} TSFRESH features for {len(tsfresh_rolling)} rows.")
    print(tsfresh_rolling.head())
    tsfresh_rolling.to_csv("tsfresh_results_3_imputed.csv", index=False)
    # # Merge to agg_full if you want:
    # agg_full = agg_full.merge(tsfresh_rolling, on=['coating_id', 'order_date'], how='left')
else:
    print("No rolling TSFRESH features extracted.")



Processing: total_fill_rate, window=3


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.23it/s]



Processing: total_chf, window=3


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.21it/s]



Processing: num_orders, window=3


...


Processing: num_industries, window=3


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.02it/s]



Processing: num_product_families, window=3


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.03it/s]


Extracted 62 TSFRESH features for 21060 rows.
   total_fill_rate_w3_value__sum_values  ...  num_product_families_w3_value__minimum
0                              6.861854  ...                                     1.0
1                             10.146605  ...                                     1.0
2                              9.010868  ...                                     1.0
3                             27.997914  ...                                     2.0
4                             25.422083  ...                                     2.0

[5 rows x 62 columns]


In [5]:
import numpy as np
from tsfresh.feature_extraction import extract_features, MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute
from sklearn.preprocessing import StandardScaler

fc_parameters = MinimalFCParameters()
window_lengths = [5]# [3,5,10,21,63]  # You can add more
base_features = [ # 'total_fill_rate']
    'total_fill_rate', 'total_chf', 'num_orders', 'num_customers',
    'num_industries', 'num_product_families'
]

def batch_rolling_windows(df, base_feature, window):
    """
    Collect all rolling window slices for all (coating_id, order_date) in one DataFrame,
    assigning a unique window_id to each for TSFRESH extraction.
    """
    all_windows = []
    for coating_id, group in tqdm(df.groupby('coating_id'), desc=f"Coating {base_feature}, win={window}"):
        group = group.sort_values('order_date').reset_index(drop=True)
        if len(group) >= window:
            for i in range(window-1, len(group)):
                window_slice = group.iloc[i-window+1:i+1][['coating_id', 'order_date', base_feature]].copy()
                # Assign unique window_id (coating_id + date of window end)
                window_slice['window_id'] = f"{coating_id}_{group.iloc[i]['order_date']}"
                window_slice['time'] = window_slice['order_date']
                window_slice['id'] = window_slice['window_id']  # For TSFRESH
                window_slice['value'] = window_slice[base_feature]
                all_windows.append(window_slice[['id', 'time', 'value']])
    if all_windows:
        return pd.concat(all_windows, ignore_index=True)
    else:
        return pd.DataFrame(columns=['id', 'time', 'value'])

all_tsfresh = []

for base_feature in base_features:
    for window in window_lengths:
        print(f"\nProcessing: {base_feature}, window={window}")
        windowed_df = batch_rolling_windows(agg_full, base_feature, window)
        if not windowed_df.empty:
            feats = extract_features(
                windowed_df,
                column_id='id',
                column_sort='time',
                column_value='value',
                default_fc_parameters=fc_parameters,
                n_jobs=4,
                disable_progressbar=False
            )
            # Extract coating_id and order_date from id
            feats = feats.reset_index()
            # After feats = feats.reset_index()
            id_col = 'id' if 'id' in feats.columns else feats.columns[0]
            feats[['coating_id', 'order_date']] = feats[id_col].astype(str).str.split('_', n=1, expand=True)
            feats['order_date'] = pd.to_datetime(feats['order_date'])
            feats = feats.drop(columns=[id_col])
            # Prefix feature names
            feats = feats.add_prefix(f"{base_feature}_w{window}_")
            feats = feats.rename(columns={f"{base_feature}_w{window}_coating_id": "coating_id",
                                         f"{base_feature}_w{window}_order_date": "order_date"})
            all_tsfresh.append(feats)

if all_tsfresh:
    tsfresh_rolling = all_tsfresh[0]
    for df in all_tsfresh[1:]:
        tsfresh_rolling = pd.merge(tsfresh_rolling, df, on=['coating_id', 'order_date'], how='outer')
    
    tsfresh_rolling.to_csv("tsfresh_results_5_not_imputed.csv", index=False)
    # Impute missing values
    numeric_cols = tsfresh_rolling.select_dtypes(include=[np.number]).columns
    # tsfresh_rolling[numeric_cols] = impute(tsfresh_rolling[numeric_cols])
    for col in numeric_cols:
        tsfresh_rolling[col] = tsfresh_rolling[col].fillna(0)
    
    tsfresh_rolling = tsfresh_rolling.sort_values(['coating_id', 'order_date']).reset_index(drop=True)
    print(f"Extracted {len(tsfresh_rolling.columns)} TSFRESH features for {len(tsfresh_rolling)} rows.")
    print(tsfresh_rolling.head())
    tsfresh_rolling.to_csv("tsfresh_results_5_imputed.csv", index=False)
    # # Merge to agg_full if you want:
    # agg_full = agg_full.merge(tsfresh_rolling, on=['coating_id', 'order_date'], how='left')
else:
    print("No rolling TSFRESH features extracted.")


Processing: total_fill_rate, window=5


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.06it/s]



Processing: total_chf, window=5


Feature Extraction: 100%|██████████| 20/20 [00:07<00:00,  2.58it/s]



Processing: num_orders, window=5


...


Processing: num_industries, window=5


Feature Extraction: 100%|██████████| 20/20 [00:08<00:00,  2.41it/s]



Processing: num_product_families, window=5


Feature Extraction: 100%|██████████| 20/20 [00:07<00:00,  2.71it/s]


Extracted 62 TSFRESH features for 20988 rows.
   total_fill_rate_w5_value__sum_values  ...  num_product_families_w5_value__minimum
0                             14.624807  ...                                     1.0
1                             34.249104  ...                                     1.0
2                             30.565413  ...                                     1.0
3                             30.790948  ...                                     2.0
4                             31.556363  ...                                     2.0

[5 rows x 62 columns]


In [6]:
import numpy as np
from tsfresh.feature_extraction import extract_features, MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute
from sklearn.preprocessing import StandardScaler

fc_parameters = MinimalFCParameters()
window_lengths = [10]# [3,5,10,21,63]  # You can add more
base_features = [ # 'total_fill_rate']
    'total_fill_rate', 'total_chf', 'num_orders', 'num_customers',
    'num_industries', 'num_product_families'
]

def batch_rolling_windows(df, base_feature, window):
    """
    Collect all rolling window slices for all (coating_id, order_date) in one DataFrame,
    assigning a unique window_id to each for TSFRESH extraction.
    """
    all_windows = []
    for coating_id, group in tqdm(df.groupby('coating_id'), desc=f"Coating {base_feature}, win={window}"):
        group = group.sort_values('order_date').reset_index(drop=True)
        if len(group) >= window:
            for i in range(window-1, len(group)):
                window_slice = group.iloc[i-window+1:i+1][['coating_id', 'order_date', base_feature]].copy()
                # Assign unique window_id (coating_id + date of window end)
                window_slice['window_id'] = f"{coating_id}_{group.iloc[i]['order_date']}"
                window_slice['time'] = window_slice['order_date']
                window_slice['id'] = window_slice['window_id']  # For TSFRESH
                window_slice['value'] = window_slice[base_feature]
                all_windows.append(window_slice[['id', 'time', 'value']])
    if all_windows:
        return pd.concat(all_windows, ignore_index=True)
    else:
        return pd.DataFrame(columns=['id', 'time', 'value'])

all_tsfresh = []

for base_feature in base_features:
    for window in window_lengths:
        print(f"\nProcessing: {base_feature}, window={window}")
        windowed_df = batch_rolling_windows(agg_full, base_feature, window)
        if not windowed_df.empty:
            feats = extract_features(
                windowed_df,
                column_id='id',
                column_sort='time',
                column_value='value',
                default_fc_parameters=fc_parameters,
                n_jobs=4,
                disable_progressbar=False
            )
            # Extract coating_id and order_date from id
            feats = feats.reset_index()
            # After feats = feats.reset_index()
            id_col = 'id' if 'id' in feats.columns else feats.columns[0]
            feats[['coating_id', 'order_date']] = feats[id_col].astype(str).str.split('_', n=1, expand=True)
            feats['order_date'] = pd.to_datetime(feats['order_date'])
            feats = feats.drop(columns=[id_col])
            # Prefix feature names
            feats = feats.add_prefix(f"{base_feature}_w{window}_")
            feats = feats.rename(columns={f"{base_feature}_w{window}_coating_id": "coating_id",
                                         f"{base_feature}_w{window}_order_date": "order_date"})
            all_tsfresh.append(feats)

if all_tsfresh:
    tsfresh_rolling = all_tsfresh[0]
    for df in all_tsfresh[1:]:
        tsfresh_rolling = pd.merge(tsfresh_rolling, df, on=['coating_id', 'order_date'], how='outer')
    
    tsfresh_rolling.to_csv("tsfresh_results_10_not_imputed.csv", index=False)
    # Impute missing values
    numeric_cols = tsfresh_rolling.select_dtypes(include=[np.number]).columns
    # tsfresh_rolling[numeric_cols] = impute(tsfresh_rolling[numeric_cols])
    for col in numeric_cols:
        tsfresh_rolling[col] = tsfresh_rolling[col].fillna(0)
    
    tsfresh_rolling = tsfresh_rolling.sort_values(['coating_id', 'order_date']).reset_index(drop=True)
    print(f"Extracted {len(tsfresh_rolling.columns)} TSFRESH features for {len(tsfresh_rolling)} rows.")
    print(tsfresh_rolling.head())
    tsfresh_rolling.to_csv("tsfresh_results_10_imputed.csv", index=False)
    # # Merge to agg_full if you want:
    # agg_full = agg_full.merge(tsfresh_rolling, on=['coating_id', 'order_date'], how='left')
else:
    print("No rolling TSFRESH features extracted.")


Processing: total_fill_rate, window=10


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.00it/s]



Processing: total_chf, window=10


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  3.05it/s]



Processing: num_orders, window=10


...


Processing: num_industries, window=10


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  2.92it/s]



Processing: num_product_families, window=10


Feature Extraction: 100%|██████████| 20/20 [00:06<00:00,  2.94it/s]


Extracted 62 TSFRESH features for 20808 rows.
   total_fill_rate_w10_value__sum_values  ...  num_product_families_w10_value__minimum
0                              43.270704  ...                                      1.0
1                              52.935870  ...                                      1.0
2                              53.448367  ...                                      1.0
3                              53.052699  ...                                      2.0
4                              53.849519  ...                                      2.0

[5 rows x 62 columns]


In [7]:
# Import all TSFRESH imputed files and merge with agg_full
import os
import pandas as pd

# List of window sizes that were processed
window_sizes = [3, 5, 10]

# Start with the original agg_full dataframe
merged_data = agg_full.copy()

print(f"Starting with agg_full: {merged_data.shape}")

# Import and merge each TSFRESH result file
for window in window_sizes:
    filename = f"tsfresh_results_{window}_imputed.csv"
    
    if os.path.exists(filename):
        print(f"\nImporting {filename}...")
        tsfresh_df = pd.read_csv(filename, parse_dates=['order_date'])
        
        print(f"  - Shape: {tsfresh_df.shape}")
        print(f"  - Columns: {len(tsfresh_df.columns)} features")
        
        # Merge with the main dataframe
        merged_data = pd.merge(
            merged_data, 
            tsfresh_df, 
            on=['coating_id', 'order_date'], 
            how='left'
        )
        
        print(f"  - After merge: {merged_data.shape}")
    else:
        print(f"Warning: {filename} not found!")

print(f"\nFinal merged dataset shape: {merged_data.shape}")
print(f"Total columns: {len(merged_data.columns)}")

# Display some info about the merged dataset
print("\nColumn breakdown:")
original_cols = len(agg_full.columns)
tsfresh_cols = len(merged_data.columns) - original_cols
print(f"  - Original agg_full columns: {original_cols}")
print(f"  - Added TSFRESH columns: {tsfresh_cols}")

# Save the final merged dataset
merged_data.to_csv("agg_full_with_tsfresh.csv", index=False)
print(f"\nSaved merged dataset to 'agg_full_with_tsfresh.csv'")

# Display first few rows to verify
print("\nFirst few rows of merged dataset:")
print(merged_data.head())

Starting with agg_full: (21132, 556)

Importing tsfresh_results_3_imputed.csv...
  - Shape: (21060, 62)
  - Columns: 62 features
  - After merge: (21132, 616)

Importing tsfresh_results_5_imputed.csv...
  - Shape: (20988, 62)
  - Columns: 62 features
  - After merge: (21132, 676)

Importing tsfresh_results_10_imputed.csv...
  - Shape: (20808, 62)
  - Columns: 62 features
  - After merge: (21132, 736)

Final merged dataset shape: (21132, 736)
Total columns: 736

Column breakdown:
  - Original agg_full columns: 556
  - Added TSFRESH columns: 180

Saved merged dataset to 'agg_full_with_tsfresh.csv'

First few rows of merged dataset:
  order_date  ...  num_product_families_w10_value__minimum
0 2023-01-06  ...                                      NaN
1 2023-01-09  ...                                      NaN
2 2023-01-10  ...                                      NaN
3 2023-01-11  ...                                      NaN
4 2023-01-20  ...                                      0.0

[5 rows